In [9]:
import tiktoken
import numpy as np

with open("/home/harsha/Projects/Astra/Optional/sample_dataset/canary_dataset.txt","r",encoding="utf-8") as f:
    content = f.read()

enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode(content)

n = len(tokens)
split_idx = int(n*0.9)

train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

train_array = np.array(train_tokens,dtype=np.uint16)
val_array  =  np.array(val_tokens,dtype=np.uint16)

train_array.tofile("train.bin")
val_array.tofile("val.bin")

print(f"Saved train.bin with {len(train_array)} tokens.")
print(f"Saved val.bin with {len(val_array)} tokens.")

Saved train.bin with 1556820 tokens.
Saved val.bin with 172981 tokens.


import the tiktoken module
we need to use the gpt 2 encodings here

now we need to encode the content to the tokens
once we did that,next we need to split the tokens for the training and testing

there is a reason we actually convert the tokens of the python list to the ,it will be explained by gemini

There are two huge, practical reasons why standard Python lists break down here and NumPy + binary files are mandatory:


### 1. Memory Bloat: 8 Bytes vs 28 Bytes Per Number

In pure Python, an integer is **not** just a raw number in memory. It is a heavy CPython heap object containing:

* A reference count (`ob_refcnt`, 8 bytes)
* A type pointer (`ob_type`, 8 bytes)
* Size metadata (8 bytes)
* The actual digit value (4-8 bytes)
* Plus the 8-byte pointer inside the Python `list` pointing to that object.

**Total cost in Python:** Around **28 to 36 bytes** per single integer.

* If you have 100 million tokens in a Python list, that takes over **3.2 GB of RAM** just to sit in memory doing nothing.

**NumPy `uint16`:**

* Has zero object wrappers.
* It packs the raw binary bytes side-by-side in continuous memory.
* Because your max token ID is 50,256, it fits cleanly in 16 bits (**exactly 2 bytes**).
* 100 million tokens in NumPy = **200 MB flat**. That is a 16x memory drop.


### 2. The Binary File (`.bin`) and `np.memmap`: Zero-RAM Slicing

If you save your data as JSON, CSV, or a text file:

1. Every time you start training, Python has to parse the text, re-create millions of integer objects, and fill up gigabytes of RAM.
2. If your dataset grows to 10 GB or 50 GB, your computer will throw an `OutOfMemoryError` before training even starts.

When you save with `tofile("train.bin")`, you dump the exact contiguous byte stream straight to disk with no formatting or headers.

This unlocks **`np.memmap`** for tomorrow's training loop:

* `np.memmap` does **not** load the file into RAM.
* It tells the Linux OS kernel: *"Map this file directly into the virtual memory address space."*
* When your data loader asks for a random batch of 64 tokens (`data[idx : idx + 64]`), the OS page cache pulls *only those specific 128 bytes* directly off the SSD into your CPU cache.
* The rest of the file stays on disk. You can stream a 100 GB dataset on a machine with only 4 GB of RAM, and loading each batch takes microseconds.

Whenever you need it later, you don't even have to "read" or "load" the whole file. You just point `np.memmap("train.bin", dtype=np.uint16, mode="r")` at the file path and start slicing immediately.